In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsentregafinal")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/clientes/clientes_raw.csv"

In [0]:
clientes_schema = StructType(fields=[
    StructField("cliente_id", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("fecha_registro", StringType(), True),
    StructField("edad", StringType(), True),
    StructField("genero", StringType(), True),
    StructField("codigo_postal", StringType(), True)
])

In [0]:
df_clientes_read = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .schema(clientes_schema)
    .csv(ruta)
)

In [0]:
display(df_clientes_read)

In [0]:
df_clientes_ingestion_date = (
    df_clientes_read
    .withColumn("ingestion_date", current_timestamp())
)

In [0]:
df_clientes_final = df_clientes_ingestion_date.select(
    "cliente_id",
    "nombre",
    "email",
    "telefono",
    "estado",
    "fecha_registro",
    "edad",
    "genero",
    "codigo_postal",
    "ingestion_date"
)

In [0]:
df_clientes_final.write.mode("overwrite").insertInto(
    f"{catalogo}.{esquema}.clientes"
)

In [0]:
display(
    spark.table(f"{catalogo}.{esquema}.clientes")
)

In [0]:
cantidad_origen = df_clientes_read.count()
cantidad_bronze = spark.table(
    f"{catalogo}.{esquema}.clientes"
).count()

print(f"Registros leídos del archivo: {cantidad_origen}")
print(f"Registros cargados en Bronze: {cantidad_bronze}")

In [0]:
%sql
SELECT *
FROM retail_dev.bronze.clientes
LIMIT 10;